In [58]:
import pandas as pd
import numpy as np

### **UCDP**
---

UCDP Georeferenced Event Dataset (GED) Global version 25.1 spanning from 1989-01 to 2024-12

In [59]:
df_ucdp = pd.read_csv('../../data/input/ucdp/GEDEvent_v25_1.csv', low_memory=False)
relevant_columns = ['country','conflict_new_id', 'date_start','date_end','best', 'dyad_new_id', 'type_of_violence',  'conflict_name', 'side_a', 'side_b']
df_ucdp = df_ucdp[relevant_columns].sort_values(['country', 'conflict_new_id', 'date_start'])
#filter by type 1 of conflict
df_ucdp = df_ucdp[df_ucdp['type_of_violence']==1].reset_index(drop=True).copy()

#join with isocode and add isocode
isocodes = pd.read_csv('../../data/input/isocodes/isocodes_appended.csv', sep = ';').rename(columns = {'alpha_3':'isocode'})
isocodes = isocodes[['name', 'isocode', 'region','sub_region']]
df_ucdp = df_ucdp.merge(isocodes, left_on='country', right_on='name', how='left')

df_ucdp

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,conflict_name,side_a,side_b,name,isocode,region,sub_region
0,Afghanistan,259,2017-07-31 00:00:00.000,2017-07-31 00:00:00.000,6,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia
1,Afghanistan,259,2021-08-26 00:00:00.000,2021-08-26 00:00:00.000,183,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia
2,Afghanistan,259,2021-08-28 00:00:00.000,2021-08-28 00:00:00.000,2,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia
3,Afghanistan,259,2021-08-29 00:00:00.000,2021-08-29 00:00:00.000,10,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia
4,Afghanistan,333,1989-01-01 00:00:00.000,1989-01-01 00:00:00.000,4,727,1,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG,Asia,Southern Asia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-30 00:00:00.000,2024-05-30 00:00:00.000,14,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN
271327,Yemen (North Yemen),16099,2024-06-13 00:00:00.000,2024-06-13 00:00:00.000,5,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN
271328,Yemen (North Yemen),16099,2024-09-10 00:00:00.000,2024-09-10 00:00:00.000,2,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN
271329,Yemen (North Yemen),16099,2024-11-12 00:00:00.000,2024-11-12 00:00:00.000,10,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN


In [60]:
df_ucdp["date_start"] = pd.to_datetime(df_ucdp["date_start"]).dt.to_period("M").dt.to_timestamp()
df_ucdp["date_end"]   = pd.to_datetime(df_ucdp["date_end"]).dt.to_period("M").dt.to_timestamp()

# Number of months in each row (inclusive)
n_months = (
    (df_ucdp["date_end"].dt.to_period("M") - df_ucdp["date_start"].dt.to_period("M"))
    .apply(lambda x: x.n + 1)
)

# Repeat rows for each month
df_expanded = df_ucdp.loc[df_ucdp.index.repeat(n_months)].copy()

month_offsets = df_expanded.groupby(level=0).cumcount()

df_expanded["date"] = (
    df_expanded["date_start"].dt.to_period("M")
    .add(month_offsets.values)
    .dt.to_timestamp()
)

df_expanded["best"] = df_expanded["best"] / n_months.repeat(n_months).values
df_expanded

,country,conflict_new_id,date_start,date_end,best,dyad_new_id,type_of_violence,conflict_name,side_a,side_b,name,isocode,region,sub_region,date
0,Afghanistan,259,2017-07-01,2017-07-01,6.0,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia,2017-07-01
1,Afghanistan,259,2021-08-01,2021-08-01,183.0,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia,2021-08-01
2,Afghanistan,259,2021-08-01,2021-08-01,2.0,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia,2021-08-01
3,Afghanistan,259,2021-08-01,2021-08-01,10.0,524,1,Iraq: Government,Government of Iraq,IS,Afghanistan,AFG,Asia,Southern Asia,2021-08-01
4,Afghanistan,333,1989-01-01,1989-01-01,4.0,727,1,Afghanistan: Government,Government of Afghanistan,Hizb-i Islami-yi Afghanistan - Khalis faction,Afghanistan,AFG,Asia,Southern Asia,1989-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271326,Yemen (North Yemen),16099,2024-05-01,2024-05-01,14.0,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN,2024-05-01
271327,Yemen (North Yemen),16099,2024-06-01,2024-06-01,5.0,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN,2024-06-01
271328,Yemen (North Yemen),16099,2024-09-01,2024-09-01,2.0,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN,2024-09-01
271329,Yemen (North Yemen),16099,2024-11-01,2024-11-01,10.0,17738,1,"United Kingdom, United States of America - Yemen","Government of United Kingdom, Government of Un...",Government of Yemen (North Yemen),Yemen (North Yemen),YEM,NaN,NaN,2024-11-01


In [61]:
#2. Aggregate to country-month level and sum the 'best' values without distinguishing countries
#-----------------------------------------------------------------------------------------------
df_country_month = (
    df_expanded
    .groupby(['isocode','country','date'], as_index=False)
    .agg(
        best=('best','sum'),
        n_conflicts=('conflict_new_id', 'nunique'),
        n_events=('dyad_new_id', 'count'),
        region = ('region','first'),
        sub_region = ('sub_region','first')    
    )
)
df_country_month = df_country_month.rename(columns={'date':'year_mo'})
df_country_month

,isocode,country,year_mo,best,n_conflicts,n_events,region,sub_region
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia,Southern Asia
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia,Southern Asia
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia,Southern Asia
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia,Southern Asia
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia,Southern Asia
...,...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,None,None
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,None,None
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa,Sub-Saharan Africa
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa,Sub-Saharan Africa


## **Instrumental Variables**

1. Distance 
2. Volume trades
3. UN voting similiraty wighted by influence veto and gdp.

### **Distance**
---

In [ ]:
df_distance = pd.read_excel('../../data/input/distance/dist_cepii.xls')

iso_fix = {
    "ANT": "CUW", 
    "PAL": "PSE",
    "ROM": "ROU",
    "TMP": "TLS",
    "ZAR": "COD",
}

df_distance["iso_o"] = df_distance["iso_o"].replace(iso_fix)
df_distance["iso_d"] = df_distance["iso_d"].replace(iso_fix)

# 1. Add South Sudan (SSD) by duplicating Sudan (SDN) dyads in pre-independence data
#---------------------------------------------------------------
# rows with SDN on at least one side
m = df_distance[["iso_o", "iso_d"]].eq("SDN").any(axis=1)

# duplicate and convert SDN -> SSD
df_ssd = df_distance.loc[m].replace({"SDN": "SSD"})

# genertes combination of SDN and SSD
df_mix = (
    df_distance.loc[(df_distance["iso_o"]=="SDN") & (df_distance["iso_d"]=="SDN")]
    .assign(iso_d="SSD")
    
)

df_mix_2 = (
    df_distance.loc[(df_distance["iso_o"]=="SDN") & (df_distance["iso_d"]=="SDN")]
    .assign(iso_o="SSD")    
)

# merge everything and drop duplicates
df_distance = (
    pd.concat([df_distance, df_ssd, df_mix, df_mix_2], ignore_index=True)
    .drop_duplicates(subset=["iso_o", "iso_d"])
    .reset_index(drop=True)
)

# 2. Add region and sub_region for isocode origin and destination
#---------------------------------------------------------------
isocodes_unique = isocodes[['isocode', 'region', 'sub_region']].dropna().drop_duplicates(subset=['isocode'])
df_distance = df_distance.merge(isocodes_unique, left_on = 'iso_o', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_o', 'sub_region':'sub_region_o'})
df_distance = df_distance.merge(isocodes_unique, left_on = 'iso_d', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_d', 'sub_region':'sub_region_d'})
df_distance = df_distance.dropna(subset=['region_o', 'region_d', 'sub_region_o', 'sub_region_d']).reset_index(drop=True)

# 3. Create conditional distance variables
#---------------------------------------------------------------
df_distance["dist_region_c"] = df_distance["dist"].where(df_distance["region_o"].eq(df_distance["region_d"]))
df_distance["dist_subregion_c"] = df_distance["dist"].where(df_distance["sub_region_o"].eq(df_distance["sub_region_d"]))

df_distance = df_distance.rename(columns={"iso_o": "isocode"})  

# 4. Aggregate to country level
#---------------------------------------------------------------
df_distance = (
    df_distance.groupby("isocode", as_index=False)[["dist_region_c", "dist_subregion_c"]]
      .mean().round(2)
)
df_distance

,isocode,dist_region_c,dist_subregion_c
0,ABW,2071.56,1880.93
1,AFG,3126.90,1795.70
2,AGO,3004.18,2760.02
3,AIA,2169.01,2056.06
4,ALB,1365.41,965.07
...,...,...,...
220,YEM,4480.09,1979.15
221,YUG,1190.14,702.99
222,ZAF,4612.81,4219.74
223,ZMB,3393.75,3071.87


### **Volume traded**
---

In [66]:
# --- read + concatenate yearly files (keep only needed cols)
trade_dir =   "../../data/input/trade/BACI_HS92_V202501"
years = range(1995, 2024)
dfs = []
for y in years:
    f = trade_dir + f"/BACI_HS92_Y{y}_V202501.csv"
    tmp = pd.read_csv(f, usecols=["t", "i", "j", "v"])
    tmp = tmp.rename(columns={"t": "year", "i": "loc1iso", "j": "loc2iso", "v": "value_export"})
    dfs.append(tmp)

trade = pd.concat(dfs, ignore_index=True)

# --- collapse (sum) by year, exporter, importer
trade = (
    trade.groupby(["year", "loc1iso", "loc2iso"], as_index=False)["value_export"]
         .sum()
)
trade

,year,loc1iso,loc2iso,value_export
0,1995,4,12,36.687
1,1995,4,20,11.060
2,1995,4,36,41.639
3,1995,4,40,421.421
4,1995,4,58,17093.855
...,...,...,...,...
838185,2023,894,826,11971.855
838186,2023,894,834,146847.794
838187,2023,894,842,192080.157
838188,2023,894,854,733.210


In [67]:
country_codes = dict(pd.read_csv("../../data/input/trade/BACI_HS92_V202501/country_codes_V202501.csv", usecols = ['country_code', 'country_iso3']).values)
trade['loc1iso'] = trade['loc1iso'].map(country_codes)
trade['loc2iso'] = trade['loc2iso'].map(country_codes)

iso_fix = {
    "ANT": "CUW", 
    "PAL": "PSE",
    "ROM": "ROU",
    "TMP": "TLS",
    "ZAR": "COD",
}

trade["loc1iso"] = trade["loc1iso"].replace(iso_fix)
trade["loc2iso"] = trade["loc2iso"].replace(iso_fix)

trade = trade.merge(isocodes_unique, left_on = 'loc1iso', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_1', 'sub_region':'sub_region_1'})
trade = trade.merge(isocodes_unique, left_on = 'loc2iso', right_on = 'isocode', how = 'left').drop(columns = ['isocode']).rename(columns = {'region':'region_2', 'sub_region':'sub_region_2'})
trade


,year,loc1iso,loc2iso,value_export,region_1,sub_region_1,region_2,sub_region_2
0,1995,AFG,DZA,36.687,Asia,Southern Asia,Africa,Northern Africa
1,1995,AFG,AND,11.060,Asia,Southern Asia,Europe,Southern Europe
2,1995,AFG,AUS,41.639,Asia,Southern Asia,Oceania,Australia and New Zealand
3,1995,AFG,AUT,421.421,Asia,Southern Asia,Europe,Western Europe
4,1995,AFG,BEL,17093.855,Asia,Southern Asia,Europe,Western Europe
...,...,...,...,...,...,...,...,...
838185,2023,ZMB,GBR,11971.855,Africa,Sub-Saharan Africa,Europe,Northern Europe
838186,2023,ZMB,TZA,146847.794,Africa,Sub-Saharan Africa,Africa,Sub-Saharan Africa
838187,2023,ZMB,USA,192080.157,Africa,Sub-Saharan Africa,Americas,Northern America
838188,2023,ZMB,BFA,733.210,Africa,Sub-Saharan Africa,Africa,Sub-Saharan Africa


In [68]:
# 3. Create conditional distance variables
#---------------------------------------------------------------
trade["value_export_region_c"] = trade["value_export"].where(trade["region_1"].eq(trade["region_2"]))
trade
trade["value_export_subregion_c"] = trade["value_export"].where(trade["sub_region_1"].eq(trade["sub_region_2"]))

trade = trade.rename(columns={"loc1iso": "isocode"})  

# 4. Aggregate to country level
#---------------------------------------------------------------
trade = (
    trade.groupby("isocode", as_index=False)
         .agg(
             value_export_region_c=("value_export_region_c", "sum"),
             value_export_subregion_c=("value_export_subregion_c", "sum"),
             total_export_world=("value_export", "sum")
         )
)

# --- shares
trade["percent_export_region_c"] = trade["value_export_region_c"] / trade["total_export_world"]
trade["percent_export_subregion_c"] = trade["value_export_subregion_c"] / trade["total_export_world"]

# --- thousands USD -> million USD (divide by 1000)
cols_musd = ["value_export_region_c", "value_export_subregion_c", "total_export_world"]
trade[cols_musd] = trade[cols_musd] / 1000

trade


,isocode,value_export_region_c,value_export_subregion_c,total_export_world,percent_export_region_c,percent_export_subregion_c
0,ABW,36183.935110,8704.570708,4.369066e+04,0.828185,0.199232
1,AFG,18072.623320,13595.311030,2.163794e+04,0.835229,0.628309
2,AGO,41010.808859,40714.715138,9.505019e+05,0.043146,0.042835
3,AIA,279.302181,190.726840,4.544570e+02,0.614584,0.419681
4,ALB,41993.301702,33244.056255,5.228000e+04,0.803238,0.635885
...,...,...,...,...,...,...
225,YEM,111308.138361,13625.641467,1.304372e+05,0.853346,0.104461
226,ZA1,0.000000,0.000000,1.579377e+05,0.000000,0.000000
227,ZAF,529021.979739,514758.821309,2.411610e+06,0.219365,0.213450
228,ZMB,39626.286120,34946.354908,2.153975e+05,0.183968,0.162241


### **Agreements**



In [14]:
df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')
df_pax.columns = df_pax.columns.str.lower()
df_pax = df_pax.rename(columns={"ucdpcon": "conflict_id"})

df_pax["dat"] = pd.to_datetime(df_pax["dat"])
df_pax["year_mo"] = df_pax["dat"].dt.to_period("M").dt.to_timestamp()

df_pax = (
    df_pax
    .set_index([c for c in df_pax.columns if c not in ['loc1iso', 'loc2iso']])
    [['loc1iso', 'loc2iso']]
    .stack()
    .reset_index()
    .drop(columns='level_278')  # removes column indicating loc1/loc2
    .rename(columns={0: 'isocode'})
)

# Replace SSD -> SDN before July 2011
mask = (df_pax["isocode"] == "SSD") & (
    (df_pax["dat"].dt.year < 2011) |
    ((df_pax["dat"].dt.year == 2011) & (df_pax["dat"].dt.month <= 6))
)
df_pax.loc[mask, "isocode"] = "SDN"

df_pax = df_pax.dropna(subset=['conflict_id'])
df_pax["conflict_id"] = df_pax["conflict_id"].astype(int)

df_pax["agreement"] = 1
df_pax["subs_agreement"] = (df_pax["stage"].str.lower() == "subpar").astype(int)
df_pax["comp_agreement"] = (df_pax["stage"].str.lower() == "subcomp").astype(int)
df_pax["cea_agreement"] = (df_pax["stage"].str.lower() == "cea").astype(int)
df_pax["cea_ceamix_agreement"] = (df_pax["stagesub"].str.lower() == "ceamix").astype(int)
df_pax["cea_ceas_agreement"] = (df_pax["stagesub"].str.lower() == "ceas").astype(int)
df_pax["cea_rel_agreement"] = (df_pax["stagesub"].str.lower() == "rel").astype(int)

#define variables to test clusterd Heterogenous TE
features_cluster_columns = ['hriimon', 'ime', 'ddrprog', 'imun', 'tjrep', 'ppsaut', 'tjmis', 
                            'protgrp', 'ppsvet', 'hriibod', 'tpsloc', 'terps', 'pol', 'epsres', 
                            'impk', 'epsoth', 'polnewtemp', 'protciv', 'hrtrinc', 'ssrddr', 
                            'ceprov', 'ppsint', 'ddrdemil', 'polpar', 'polps', 'hrbor', 'hrdem',
                            'medlog', 'tjvet', 'medsubs', 'epsfis', 'tpssub', 'tpsoth', 'eleccomm', 
                            'eps', 'ssrpol', 'medgov', 'imref', 'tpsaut', 'med', 'hrii', 'cegen', 'ssrgua',
                            'ssrarm', 'tjcou', 'ppsoro', 'tjamban', 'tjvic', 'tjmech', 'polnewind', 'mpsme', 
                            'mpsjt', 'imoth', 'ppsothpr', 'ppsex', 'hrmob', 'ele', 'hrni', 'civso', 'pubad', 
                            'juscr', 'mps', 'prot', 'mpspro']


df_agreements = (
    df_pax
    .groupby(["isocode", "year_mo"], as_index=False)
    .agg(
        agreement=("agreement", "max"),  # al menos un acuerdo ese mes
        comp_agreement=("comp_agreement", "max"),  # al menos un SubComp
        subs_agreement=("subs_agreement", "max"),
        cea_agreement = ('cea_agreement', "max"), 
        cea_ceamix_agreement = ('cea_ceamix_agreement', "max"),
        cea_ceas_agreement = ('cea_ceas_agreement', "max"),
        cea_rel_agreement = ('cea_rel_agreement', "max"),
        ce=('ce','max'),
        **{var: (var, "max") for var in features_cluster_columns}
    )
)
df_agreements

/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/1551532066.py:1: DtypeWarning: Columns (23,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pax = pd.read_csv('../../data/input/pax/pax_data_2144_agreements_v9_10.csv')
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/1551532066.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/1551532066.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.cop

,isocode,year_mo,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce
0,AFG,1992-04-01,1,0,1,0,0,0,0,0
1,AFG,1993-03-01,1,0,1,0,0,0,0,2
2,AFG,1999-07-01,1,0,0,0,0,0,0,1
3,AFG,2001-12-01,1,0,1,0,0,0,0,0
4,AFG,2002-01-01,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
1408,ZAF,1992-09-01,1,0,0,0,0,0,0,0
1409,ZAF,1992-11-01,1,0,1,0,0,0,0,0
1410,ZAF,1993-11-01,1,1,0,0,0,0,0,0
1411,ZAF,1994-02-01,1,0,1,0,0,0,0,0


In [15]:
df_panel = df_country_month.merge(
    df_agreements,
    on=["isocode", "year_mo"],
    how="left"
)
df_panel[["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns] = df_panel[
    ["agreement", "comp_agreement", 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 'cea_ceamix_agreement', 'cea_rel_agreement', 'ce']+features_cluster_columns
].fillna(0).astype(int)

df_panel['isocode_num'] = df_panel['isocode'].astype('category').cat.codes + 1
df_panel['region_num'] = df_panel['region'].astype('category').cat.codes + 1

df_panel

,isocode,country,year_mo,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,isocode_num,region_num
0,AFG,Afghanistan,1989-01-01,693.750000,1,13,Asia,0,0,0,0,0,0,0,0,1,3
1,AFG,Afghanistan,1989-02-01,162.750000,1,18,Asia,0,0,0,0,0,0,0,0,1,3
2,AFG,Afghanistan,1989-03-01,1745.750000,1,21,Asia,0,0,0,0,0,0,0,0,1,3
3,AFG,Afghanistan,1989-04-01,495.750000,1,28,Asia,0,0,0,0,0,0,0,0,1,3
4,AFG,Afghanistan,1989-05-01,441.000000,1,18,Asia,0,0,0,0,0,0,0,0,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11597,YEM,Yemen (North Yemen),2024-11-01,26.166667,2,8,Middle East,0,0,0,0,0,0,0,0,107,5
11598,YEM,Yemen (North Yemen),2024-12-01,61.166667,3,23,Middle East,0,0,0,0,0,0,0,0,107,5
11599,ZAF,South Africa,1989-02-01,0.000000,1,1,Africa,0,0,0,0,0,0,0,0,108,1
11600,ZAF,South Africa,1989-04-01,359.000000,1,55,Africa,0,0,0,0,0,0,0,0,108,1


## **Expand the panel**

Exapnd the panel from 1989-01 to 2024-12 monthly.
This is done after mergin the agreements, so we include only the interventions that occured in violent periods of the conflicts.


In [16]:
#3. Fill missing months with 0 fatalities to expand to a balanced panel
#-----------------------------------------------------------------------------------------------
# Global dataset range (for example)
global_start = df_ucdp.date_start.min()
global_end   = df_ucdp.date_end.max()

iso_to_country = df_panel.drop_duplicates('isocode').set_index('isocode')['country'].to_dict()

# crear grid de país–mes
isocodes = df_panel['isocode'].unique()
all_months = pd.date_range(global_start, global_end, freq='MS')

import itertools
full_index = pd.DataFrame(itertools.product(isocodes, all_months), columns=['isocode', 'year_mo'])

# merge con tus datos y rellenar ceros
df_panel = full_index.merge(df_panel, on=['isocode', 'year_mo'], how='left')
df_panel['country'] = df_panel['isocode'].map(iso_to_country)
df_panel['real_observation'] = np.where(df_panel['best'].notna(), 1, 0)


# Fill in missing values
columns_to_fill = ['best', 'agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement', 'ce'] + features_cluster_columns
for i in columns_to_fill:
    df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)

df_panel['log_best'] = np.log1p(df_panel['best'])

columns_to_ffill_bfill = ['isocode', 'country', 'isocode_num', 'region','region_num', 'type_of_violence']

for i in columns_to_ffill_bfill:
    df_panel[i] = df_panel.groupby('isocode')[i].transform(lambda x: x.ffill().bfill())

df_panel


/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df_panel[i] = df_panel.groupby('isocode')[i].fillna(0)
/var/folders/fz/1mp_qssx1z546qc9z63nq3rm0000gn/T/ipykernel_70589/2392698621.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to

,isocode,year_mo,country,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,cea_agreement,cea_ceamix_agreement,cea_ceas_agreement,cea_rel_agreement,ce,isocode_num,region_num,real_observation,log_best
0,AFG,1989-01-01,Afghanistan,693.75,1.0,13.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.543552
1,AFG,1989-02-01,Afghanistan,162.75,1.0,18.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,5.098341
2,AFG,1989-03-01,Afghanistan,1745.75,1.0,21.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,7.465512
3,AFG,1989-04-01,Afghanistan,495.75,1.0,28.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.208087
4,AFG,1989-05-01,Afghanistan,441.00,1.0,18.0,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1,6.091310
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46652,ZAF,2024-09-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46653,ZAF,2024-10-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000
46654,ZAF,2024-11-01,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,108.0,1.0,0,0.000000


### **First treatment date per conflict**
---


In [17]:
df_panel['year_mo'] = pd.to_datetime(df_panel['year_mo'], format='%Y-%m').dt.to_period('M')
base_date = df_panel['year_mo'].min()
#numeric month index starting from 1
df_panel['year_mo_numeric'] = (df_panel["year_mo"] - base_date).apply(lambda x: x.n+1)

for i in ['agreement', 'comp_agreement','subs_agreement', 'cea_agreement','cea_ceas_agreement', 'cea_ceamix_agreement','cea_rel_agreement']:
    #create the group variable for first treatment date
    df_panel[f'first_{i}_date'] = (
    df_panel[df_panel[i] == 1]
    .groupby('isocode')['year_mo_numeric']
    .transform('min')
    )

    df_panel[f'first_{i}_date'] = (
    df_panel.groupby('isocode')[f'first_{i}_date']
    .transform(lambda x: x.ffill().bfill())
    )
    #Variable gvar: 0 = never treated, positive = month of the first treatment
    df_panel[f'first_{i}_date'] = df_panel[f'first_{i}_date'].fillna(0).astype(int)

    #Dummy: indicator for treated units by agreements
    df_panel[f'treated_{i}'] = (
    df_panel.groupby('isocode')[i]
    .transform('max')
    )

df_panel

,isocode,year_mo,country,best,n_conflicts,n_events,region,agreement,comp_agreement,subs_agreement,...,first_subs_agreement_date,treated_subs_agreement,first_cea_agreement_date,treated_cea_agreement,first_cea_ceas_agreement_date,treated_cea_ceas_agreement,first_cea_ceamix_agreement_date,treated_cea_ceamix_agreement,first_cea_rel_agreement_date,treated_cea_rel_agreement
0,AFG,1989-01,Afghanistan,693.75,1.0,13.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
1,AFG,1989-02,Afghanistan,162.75,1.0,18.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
2,AFG,1989-03,Afghanistan,1745.75,1.0,21.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
3,AFG,1989-04,Afghanistan,495.75,1.0,28.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
4,AFG,1989-05,Afghanistan,441.00,1.0,18.0,Asia,0.0,0.0,0.0,...,40,1.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46651,ZAF,2024-08,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46652,ZAF,2024-09,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46653,ZAF,2024-10,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
46654,ZAF,2024-11,South Africa,0.00,NaN,NaN,Africa,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [ ]:
df_panel.columns = df_panel.columns.str.replace(' ', '_').str.replace('-', '_').str.replace('.', '_').str.replace('|', '_')
df_panel['best'] = df_panel['best'].astype(float)
df_panel['log_best'] = df_panel['log_best'].astype(float)
df_panel

### **Treatment variables**
---

In [ ]:
df_panel.year_mo = df_panel.year_mo.astype(str)

In [ ]:
# Define Conflict Active
#================================================================================

# Define any_violence: 1 if there were fatalities, 0 otherwise
df_panel['any_violence'] = (df_panel['log_best'] > 0).astype(int)

def gen_since(series: pd.Series):
    #count the number of consecutive months since last violence
    series = (series == False)
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()
    result[groups == 0] = np.nan
    return result

def gen_until(series: pd.Series) -> pd.Series:
    #count the number of consecutive months until next violence
    series = series[::-1]
    series = series == 0
    groups = (series == 0).cumsum()
    result = series.groupby(groups).cumsum()[::-1]
    result[groups == 0] = np.nan
    return result

# Apply function within each conflict panel
df_panel['since_any_violence'] = df_panel.groupby('isocode')['any_violence'].transform(gen_since)
df_panel.year_mo = df_panel.year_mo.astype(str)
df_panel['year'] = df_panel['year_mo'].str[:4].astype(int)

# Define conflict_active according to the 24-month rule (and special 12-month rule for 1990)
df_panel['conflict_active'] = np.where(
    (df_panel['since_any_violence'] < 24) |
    ((df_panel['since_any_violence'] < 12) & (df_panel['year']<= 1990)),
    1, 0
)

#create agreements associated with conflict active
# ================================================================================

for i in ['agreement', 'comp_agreement', 'subs_agreement', 'cea_agreement', 'cea_ceas_agreement', 
          'cea_rel_agreement', 'cea_ceamix_agreement', 'ce']:
    if i != 'ce':
        df_panel[f'{i}'] = np.where(
            (df_panel[f'{i}'] == 1) & (df_panel['conflict_active'] == 1),
            1, 0
        )
    else:
        df_panel[f'{i}asefire_mentions'] = np.where(
            (df_panel[f'{i}'] >= 1) & (df_panel['conflict_active'] == 1),
            1, 0)


df_panel['treated_ceasfire_mentions'] = (df_panel.groupby('isocode')['ceasefire_mentions'].transform('max'))

#Define definitive treatment variable
#--------------------------------------------------------------------------------
#combine cea_agreementes with ceasefire_mentions
df_panel['ceasfire_agreements_mentions'] = np.where(
    (df_panel['cea_agreement'] == 1) | (df_panel['ceasefire_mentions'] == 1),
    1,0
)
df_panel['treated_ceasfire_agreements_mentions'] = (df_panel.groupby('isocode')['ceasfire_agreements_mentions'].transform('max'))
df_panel

### **Treatment without ceasefire mentions**

In [ ]:
# months to next ceasefire mention / agreement (por conflicto)
df_panel['until_ce_mentions'] = df_panel.groupby('isocode')['ceasefire_mentions'].transform(gen_until)
df_panel['ce_mentions_active'] = (df_panel['until_ce_mentions'] >= 6).astype(int)

df_panel['until_cea_agreement'] = df_panel.groupby('isocode')['cea_agreement'].transform(gen_until)
df_panel['cea_agreement_active'] = (df_panel['until_cea_agreement'] >= 6).astype(int)

active_6m = (df_panel['ce_mentions_active'].eq(1) & df_panel['cea_agreement_active'].eq(1))


# ============================================================
# Generate no ceasefire variables for agreement, comp_agreement, subs_agreement
# ============================================================

types = ['agreement', 'comp_agreement', 'subs_agreement']

for t in types:
    base = f'{t}_no_ceasefire'
    treated = f'treated_{t}s_no_ceasefire'
    var6m = f'{t}_no_ceasefire_mentions_agreement_6m'

    # t == 1 AND ce <= 0 AND cea_agreement == 0
    df_panel[base] = (
        df_panel[t].eq(1) &
        df_panel['ce'].le(0) &
        df_panel['cea_agreement'].eq(0)
    ).astype(int)

    # max per conflict
    df_panel[treated] = df_panel.groupby('isocode')[base].transform('max')

    # (active 6m) AND base
    df_panel[var6m] = (active_6m & df_panel[base].eq(1)).astype(int)

In [9]:
df_panel.to_csv('../../data/output/country_level/country_panel.csv')